# Named Entity Recognition and Classification with spaCy

This notebook implements the NERC part of the Text Mining project. It uses the provided token-level test set, runs spaCy's pretrained English NER model, converts spaCy entity labels to the BIO labels used in the test file, and evaluates the predictions quantitatively and qualitatively.

## Task and Approach

**Task:** extract named entities from each sentence and classify them as `PER`, `ORG`, `LOC`, `MISC`, or `O` using BIO notation.

**System:** spaCy `en_core_web_sm`, a pretrained English NLP pipeline. The model was trained on OntoNotes-style entity labels, so its labels are mapped to the project's CoNLL-style labels:

- `PERSON` -> `PER`
- `ORG` -> `ORG`
- `GPE`, `LOC`, `FAC` -> `LOC`
- labels such as `NORP`, `LANGUAGE`, `WORK_OF_ART`, `PRODUCT`, `EVENT`, `LAW` -> `MISC`

The test set is already tokenized. To keep the evaluation fair at token level, the notebook creates spaCy `Doc` objects from the provided tokens and then applies the spaCy pipeline to those tokens.

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd
import spacy
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from spacy.tokens import Doc

DATA_PATH = Path("test_sets/NER-test.tsv")
OUTPUT_PATH = Path("outputs/nerc_spacy_predictions.csv")

nlp = spacy.load("en_core_web_sm")
print("spaCy version:", spacy.__version__)
print("Pipeline:", nlp.pipe_names)

spaCy version: 3.8.11
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## Load and Inspect the Test Set

The data contains one token per row, with a sentence id, token id, token text, and gold BIO NER tag.

In [3]:
ner_df = pd.read_csv(DATA_PATH, sep="\t")
ner_df.head(12)

,sentence id,token id,token,BIO NER tag
0,0,0,It,O
1,0,1,took,O
2,0,2,eight,O
3,0,3,years,O
4,0,4,for,O
5,0,5,Warner,B-ORG
6,0,6,Brothers,I-ORG
7,0,7,to,O
8,0,8,recover,O
9,0,9,from,O


In [4]:
required_columns = {"sentence id", "token id", "token", "BIO NER tag"}
missing_columns = required_columns.difference(ner_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

sentences = []
for sentence_id, group in ner_df.sort_values(["sentence id", "token id"]).groupby("sentence id", sort=True):
    tokens = group["token"].astype(str).tolist()
    gold_tags = group["BIO NER tag"].astype(str).tolist()
    sentences.append({
        "sentence_id": int(sentence_id),
        "tokens": tokens,
        "gold_tags": gold_tags,
        "sentence": " ".join(tokens),
    })

gold_tags_flat = [tag for sentence in sentences for tag in sentence["gold_tags"]]

print(f"Number of sentences: {len(sentences)}")
print(f"Number of tokens: {len(gold_tags_flat)}")
print("Gold label distribution:")
for label, count in Counter(gold_tags_flat).most_common():
    print(f"{label:7s} {count:3d}")

Number of sentences: 10
Number of tokens: 214
Gold label distribution:
O       183
I-PER     8
B-PER     6
B-ORG     4
B-LOC     4
I-ORG     3
B-MISC    3
I-LOC     2
I-MISC    1


The distribution is highly imbalanced: most tokens are `O`, while actual entity labels are much rarer. For this reason, the macro averages and entity-level results are more informative than accuracy alone.

## Run spaCy and Convert to BIO Tags

In [5]:
SPACY_TO_PROJECT_LABEL = {
    "PERSON": "PER",
    "ORG": "ORG",
    "GPE": "LOC",
    "LOC": "LOC",
    "FAC": "LOC",
    "NORP": "MISC",
    "LANGUAGE": "MISC",
    "PRODUCT": "MISC",
    "EVENT": "MISC",
    "WORK_OF_ART": "MISC",
    "LAW": "MISC",
}


def run_spacy_on_tokens(tokens):
    """Run the existing spaCy pipeline while preserving the provided tokenization."""
    doc = Doc(nlp.vocab, words=tokens)
    for _, pipe in nlp.pipeline:
        doc = pipe(doc)
    return doc


def doc_to_bio_tags(doc):
    tags = ["O"] * len(doc)
    for ent in doc.ents:
        project_label = SPACY_TO_PROJECT_LABEL.get(ent.label_)
        if project_label is None:
            continue
        tags[ent.start] = f"B-{project_label}"
        for token_i in range(ent.start + 1, ent.end):
            tags[token_i] = f"I-{project_label}"
    return tags


def bio_entities(tags):
    """Return strict BIO entities as (start, end, label), where end is exclusive."""
    entities = []
    start = None
    label = None

    for i, tag in enumerate(tags + ["O"]):
        if tag == "O" or tag == "nan":
            prefix, current_label = "O", None
        else:
            prefix, current_label = tag.split("-", 1)

        starts_new_entity = prefix == "B" or (prefix == "I" and current_label != label)
        closes_entity = start is not None and (prefix == "O" or starts_new_entity)

        if closes_entity:
            entities.append((start, i, label))
            start = None
            label = None

        if starts_new_entity:
            start = i
            label = current_label

    return entities


prediction_rows = []
sentence_level_rows = []

for sentence in sentences:
    doc = run_spacy_on_tokens(sentence["tokens"])
    predicted_tags = doc_to_bio_tags(doc)
    sentence["predicted_tags"] = predicted_tags
    sentence["spacy_entities"] = [(ent.text, ent.label_) for ent in doc.ents]

    for token_id, (token, gold_tag, predicted_tag) in enumerate(zip(sentence["tokens"], sentence["gold_tags"], predicted_tags)):
        prediction_rows.append({
            "sentence_id": sentence["sentence_id"],
            "token_id": token_id,
            "token": token,
            "gold_tag": gold_tag,
            "predicted_tag": predicted_tag,
            "correct": gold_tag == predicted_tag,
        })

    gold_entities = bio_entities(sentence["gold_tags"])
    predicted_entities = bio_entities(predicted_tags)
    sentence_level_rows.append({
        "sentence_id": sentence["sentence_id"],
        "sentence": sentence["sentence"],
        "gold_entities": gold_entities,
        "predicted_entities": predicted_entities,
        "spacy_raw_entities": sentence["spacy_entities"],
    })

predictions_df = pd.DataFrame(prediction_rows)
sentence_predictions_df = pd.DataFrame(sentence_level_rows)
predictions_df.head(20)

,sentence_id,token_id,token,gold_tag,predicted_tag,correct
0,0,0,It,O,O,True
1,0,1,took,O,O,True
2,0,2,eight,O,O,True
3,0,3,years,O,O,True
4,0,4,for,O,O,True
5,0,5,Warner,B-ORG,B-ORG,True
6,0,6,Brothers,I-ORG,I-ORG,True
7,0,7,to,O,O,True
8,0,8,recover,O,O,True
9,0,9,from,O,O,True


In [6]:
OUTPUT_PATH.parent.mkdir(exist_ok=True)
predictions_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved token-level predictions to {OUTPUT_PATH}")

Saved token-level predictions to outputs/nerc_spacy_predictions.csv


## Quantitative Evaluation

In [7]:
label_order = ["B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC", "O"]
gold_tags = predictions_df["gold_tag"].tolist()
predicted_tags = predictions_df["predicted_tag"].tolist()

print(classification_report(gold_tags, predicted_tags, labels=label_order, zero_division=0))

              precision    recall  f1-score   support

       B-PER       0.67      0.67      0.67         6
       I-PER       1.00      0.75      0.86         8
       B-ORG       0.67      0.50      0.57         4
       I-ORG       0.75      1.00      0.86         3
       B-LOC       1.00      1.00      1.00         4
       I-LOC       1.00      1.00      1.00         2
      B-MISC       0.75      1.00      0.86         3
      I-MISC       1.00      1.00      1.00         1
           O       0.99      0.99      0.99       183

    accuracy                           0.97       214
   macro avg       0.87      0.88      0.87       214
weighted avg       0.97      0.97      0.97       214



In [8]:
entity_labels = ["PER", "ORG", "LOC", "MISC"]
gold_entity_set = set()
predicted_entity_set = set()

for sentence in sentences:
    for entity in bio_entities(sentence["gold_tags"]):
        gold_entity_set.add((sentence["sentence_id"],) + entity)
    for entity in bio_entities(sentence["predicted_tags"]):
        predicted_entity_set.add((sentence["sentence_id"],) + entity)

true_positives = gold_entity_set & predicted_entity_set
false_positives = predicted_entity_set - gold_entity_set
false_negatives = gold_entity_set - predicted_entity_set

precision = len(true_positives) / len(predicted_entity_set) if predicted_entity_set else 0
recall = len(true_positives) / len(gold_entity_set) if gold_entity_set else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print(f"Strict entity-level precision: {precision:.3f}")
print(f"Strict entity-level recall:    {recall:.3f}")
print(f"Strict entity-level F1:        {f1:.3f}")
print(f"Gold entities: {len(gold_entity_set)}")
print(f"Predicted entities: {len(predicted_entity_set)}")
print(f"Correct entities: {len(true_positives)}")

Strict entity-level precision: 0.765
Strict entity-level recall:    0.765
Strict entity-level F1:        0.765
Gold entities: 17
Predicted entities: 17
Correct entities: 13


In [9]:
cm = confusion_matrix(gold_tags, predicted_tags, labels=label_order)
confusion_df = pd.DataFrame(cm, index=[f"gold_{label}" for label in label_order], columns=[f"pred_{label}" for label in label_order])
confusion_df

,pred_B-PER,pred_I-PER,pred_B-ORG,pred_I-ORG,pred_B-LOC,pred_I-LOC,pred_B-MISC,pred_I-MISC,pred_O
gold_B-PER,4,0,0,0,0,0,0,0,2
gold_I-PER,2,6,0,0,0,0,0,0,0
gold_B-ORG,0,0,2,1,0,0,1,0,0
gold_I-ORG,0,0,0,3,0,0,0,0,0
gold_B-LOC,0,0,0,0,4,0,0,0,0
gold_I-LOC,0,0,0,0,0,2,0,0,0
gold_B-MISC,0,0,0,0,0,0,3,0,0
gold_I-MISC,0,0,0,0,0,0,0,1,0
gold_O,0,0,1,0,0,0,0,0,182


## Qualitative Error Analysis

The next cells show sentence-level examples where spaCy's prediction differs from the gold annotation. These examples can be used directly in the poster discussion.

In [10]:
def format_entities(tokens, entities):
    if not entities:
        return "None"
    formatted = []
    for start, end, label in entities:
        formatted.append(f"{' '.join(tokens[start:end])} ({label})")
    return "; ".join(formatted)


error_examples = []
for sentence in sentences:
    if sentence["gold_tags"] != sentence["predicted_tags"]:
        error_examples.append({
            "sentence_id": sentence["sentence_id"],
            "sentence": sentence["sentence"],
            "gold": format_entities(sentence["tokens"], bio_entities(sentence["gold_tags"])),
            "predicted": format_entities(sentence["tokens"], bio_entities(sentence["predicted_tags"])),
            "spacy_raw": sentence["spacy_entities"],
        })

error_examples_df = pd.DataFrame(error_examples)
display(error_examples_df)

,sentence_id,sentence,gold,predicted,spacy_raw
0,1,All the New York University students love this...,New York University (ORG); Soho (LOC),the New York University (ORG); Soho (LOC),"[(the New York University, ORG), (Soho, LOC)]"
1,6,My husband and I moved to Amsterdam 6 years ag...,Amsterdam (LOC); Blauwbrug (ORG),Amsterdam (LOC); Blauwbrug (MISC),"[(Amsterdam, GPE), (6 years ago, DATE), (Blauw..."
2,7,Dame Maggie Smith performed her role excellent...,Dame Maggie Smith (PER),Maggie Smith (PER),"[(Maggie Smith, PERSON)]"
3,8,The new movie by Mr. Kruno was shot in New Yor...,Mr. Kruno (PER); New York (LOC); Los Angeles (...,Kruno (PER); New York (LOC); Los Angeles (LOC),"[(Kruno, PERSON), (New York, GPE), (Los Angele..."


In [11]:
def describe_entity_set(entity_set, source_sentences):
    examples = []
    by_id = {sentence["sentence_id"]: sentence for sentence in source_sentences}
    for sentence_id, start, end, label in sorted(entity_set):
        sentence = by_id[sentence_id]
        examples.append({
            "sentence_id": sentence_id,
            "entity": " ".join(sentence["tokens"][start:end]),
            "label": label,
            "sentence": sentence["sentence"],
        })
    return pd.DataFrame(examples)

print("False positives: entities predicted by spaCy but not in gold")
display(describe_entity_set(false_positives, sentences))

print("False negatives: gold entities missed by spaCy")
display(describe_entity_set(false_negatives, sentences))

False positives: entities predicted by spaCy but not in gold


,sentence_id,entity,label,sentence
0,1,the New York University,ORG,All the New York University students love this...
1,6,Blauwbrug,MISC,My husband and I moved to Amsterdam 6 years ag...
2,7,Maggie Smith,PER,Dame Maggie Smith performed her role excellent...
3,8,Kruno,PER,The new movie by Mr. Kruno was shot in New Yor...


False negatives: gold entities missed by spaCy


,sentence_id,entity,label,sentence
0,1,New York University,ORG,All the New York University students love this...
1,6,Blauwbrug,ORG,My husband and I moved to Amsterdam 6 years ag...
2,7,Dame Maggie Smith,PER,Dame Maggie Smith performed her role excellent...
3,8,Mr. Kruno,PER,The new movie by Mr. Kruno was shot in New Yor...


## Discussion for the Poster

**Strengths.** spaCy performs best on common, conventional entity types such as people and locations because these patterns are frequent in the pretrained model's training data. Multi-token proper names such as `New York` and person names are often found when they look like standard named entities.

**Main sources of error.** The test set uses CoNLL-style labels, while spaCy uses OntoNotes labels. Some differences are therefore annotation differences rather than purely model mistakes. For example, nationality/adjectival expressions such as `Italian`, `African American`, or `English` are mapped from spaCy's `NORP`/`LANGUAGE`-like categories to `MISC`, but these categories are not always predicted consistently. Organization and location boundaries can also differ, especially for names that are ambiguous in review text, such as restaurants, universities, companies, and place names.

**Limitations.** The model is used out of the box and is not trained on the project data. The test set is small, so each missed entity has a visible effect on scores. The domain also mixes movie, book, and restaurant reviews, which differs from the news-style data often used for NER pretraining.

**Possible improvements.** With more time, the system could be improved by fine-tuning spaCy on CoNLL-style data, adding an `EntityRuler` for predictable names in this domain, using a larger spaCy model, and applying a more careful mapping between OntoNotes and CoNLL labels.